In [ ]:
# %% === Install Required Packages ===
!pip install --quiet pandas numpy matplotlib seaborn scikit-learn torch xgboost joblib tqdm pytorch-tabnet nltk textstat emoji lightgbm

# %% === Imports ===
import pandas as pd, numpy as np, re, os, time, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import (
    RandomForestRegressor, GradientBoostingRegressor, 
    ExtraTreesRegressor, HistGradientBoostingRegressor
)
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import BayesianRidge, Ridge
from sklearn.svm import LinearSVR
import torch
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import StackingRegressor
from sklearn.neural_network import MLPRegressor
import xgboost as xgb
from lightgbm import LGBMRegressor
import joblib
from pytorch_tabnet.tab_model import TabNetRegressor
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import textstat
import emoji

# Download VADER lexicon
nltk.download('vader_lexicon')

# === Config ===
pd.options.mode.chained_assignment = None  # Suppress pandas SettingWithCopyWarning
torch.manual_seed(42)
np.random.seed(42)

DATASET_PATH = "../../../Dataset/Mohamed_Dataset/snapshots/youtube_dataset_snapshot_20251028_144628.csv"

def train_tabular_model():
    # Load Data
    data = pd.read_csv(DATASET_PATH, low_memory=False)
    data['title'] = data['title'].astype(str)
    data = data[~data['title'].str.contains('#', regex=False, na=False)]

    # Feature Engineering
    data['publishedAt'] = pd.to_datetime(data['publishedAt'], errors='coerce')
    data['pub_year'] = data['publishedAt'].dt.year
    data['pub_month'] = data['publishedAt'].dt.month
    data['pub_day'] = data['publishedAt'].dt.day
    data['pub_weekday'] = data['publishedAt'].dt.weekday
    data['pub_hour'] = data['publishedAt'].dt.hour
    data['days_since_first_video'] = (data['publishedAt'] - data['publishedAt'].min()).dt.days

    def duration_to_seconds(d):
        if not isinstance(d, str): return 0
        h = int(re.search(r'(\d+)H', d).group(1)) if 'H' in d else 0
        m = int(re.search(r'(\d+)M', d).group(1)) if 'M' in d else 0
        s = int(re.search(r'(\d+)S', d).group(1)) if 'S' in d else 0
        return h * 3600 + m * 60 + s

    data['duration_sec'] = data['duration'].apply(duration_to_seconds)
    key_features = ['viewCount', 'likeCount', 'commentCount', 'pub_year']
    data.dropna(subset=key_features, inplace=True)
    data.drop_duplicates(subset='video_id', inplace=True, ignore_index=True)
    data = data[data['viewCount'] >= 1000]
    data['log_views'] = np.log1p(data['viewCount'])

    # Train-Test Split
    train_df, temp_df = train_test_split(data, test_size=0.3, random_state=42)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

    # Preprocessing
    if 'channel_id' in train_df.columns:
        ch_stats = train_df.groupby('channel_id')['viewCount'].agg(['mean', 'median', 'std', 'count']).fillna(0)
        ch_stats.columns = [f'channel_{c}_views' for c in ch_stats.columns]
        train_df = train_df.merge(ch_stats, on='channel_id', how='left')
        val_df = val_df.merge(ch_stats, on='channel_id', how='left')
        test_df = test_df.merge(ch_stats, on='channel_id', how='left')
        for col in ch_stats.columns:
            val_df[col] = val_df[col].fillna(0)
            test_df[col] = test_df[col].fillna(0)
    else:
        for c in ['mean', 'median', 'std', 'count']:
            col_name = f'channel_{c}_views'
            train_df[col_name] = 0
            val_df[col_name] = 0
            test_df[col_name] = 0

    numeric_features = [
        'duration_sec','pub_year','pub_month','pub_day','pub_weekday','pub_hour',
        'days_since_first_video','likeCount','commentCount',
        'channel_mean_views','channel_median_views','channel_std_views',
        'channel_count_views'
    ]
    cat_cols = [c for c in ['definition','caption','category'] if c in data.columns]

    train_df = pd.get_dummies(train_df, columns=cat_cols, drop_first=True)
    val_df = pd.get_dummies(val_df, columns=cat_cols, drop_first=True)
    test_df = pd.get_dummies(test_df, columns=cat_cols, drop_first=True)

    train_cols = train_df.columns
    val_df = val_df.reindex(columns=train_cols, fill_value=0)
    test_df = test_df.reindex(columns=train_cols, fill_value=0)

    encoded_cat_cols = [c for c in train_df.columns if any(col in c for col in cat_cols)]
    feature_cols = numeric_features + encoded_cat_cols

    X_train = train_df[feature_cols].fillna(0)
    X_val = val_df[feature_cols].fillna(0)
    X_test = test_df[feature_cols].fillna(0)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)

    y_train = train_df['log_views']
    y_val = val_df['log_views']
    y_test = test_df['log_views']

    # Train and Evaluate Models
    models = {
        "GradientBoosting": GradientBoostingRegressor(n_estimators=2000, learning_rate=0.05, max_depth=6, subsample=0.8, min_samples_leaf=3, random_state=42)
    }

    results = []
    for name, model in models.items():
        model.fit(X_train, y_train)
        preds_test_log = model.predict(X_test)
        preds_test = np.expm1(np.clip(preds_test_log, -20, 20))
        y_test_actual = np.expm1(y_test)
        
        test_r2 = r2_score(y_test_actual, preds_test)
        test_mae = mean_absolute_error(y_test_actual, preds_test)
        test_mse = mean_squared_error(y_test_actual, preds_test)
        median_error = np.median(np.abs((preds_test - y_test_actual) / (y_test_actual + 1e-9)) * 100)
        
        results.append({
            'Model': f'Tabular Only ({name})',
            'R2': test_r2,
            'MAE': test_mae,
            'MSE': test_mse,
            'Median % Error': median_error
        })
    return results[0]

def train_title_model():
    # Load Data
    data = pd.read_csv(DATASET_PATH, low_memory=False)
    data = data[['title', 'viewCount']].copy()
    data['title'] = data['title'].astype(str)
    data = data[~data['title'].str.contains('#', regex=False, na=False)]
    data = data[data['title'].str.strip() != '']
    data['log_views'] = np.log1p(data['viewCount'])

    # Feature Engineering
    sid = SentimentIntensityAnalyzer()
    data['sentiment'] = data['title'].apply(lambda x: sid.polarity_scores(x)['compound'])
    data['readability'] = data['title'].apply(textstat.flesch_reading_ease)
    data['title_length'] = data['title'].apply(len)
    data['word_count'] = data['title'].apply(lambda x: len(x.split()))
    data['uppercase_word_count'] = data['title'].apply(lambda x: len([w for w in x.split() if w.isupper()]))
    data['emoji_count'] = data['title'].apply(emoji.emoji_count)
    data['punctuation_count'] = data['title'].apply(lambda x: len([c for c in x if c in '.,?!:;']))
    data['has_question_mark'] = data['title'].apply(lambda x: '?' in x).astype(int)
    data['has_exclamation_mark'] = data['title'].apply(lambda x: '!' in x).astype(int)

    vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
    X_tfidf = vectorizer.fit_transform(data['title'])

    numeric_features = ['sentiment','readability','title_length','word_count','uppercase_word_count',
                        'emoji_count','punctuation_count','has_question_mark','has_exclamation_mark']
    scaler = StandardScaler()
    X_numeric = scaler.fit_transform(data[numeric_features])

    X = hstack([X_tfidf, X_numeric])
    y = data['log_views']

    # Train-Test Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Train and Evaluate Models
    models = {
        "RandomForest": RandomForestRegressor(n_estimators=200, max_depth=None, random_state=42, n_jobs=-1)
    }

    results = []
    for name, model in models.items():
        model.fit(X_train, y_train)
        preds_test_log = model.predict(X_test)
        preds_test = np.expm1(np.clip(preds_test_log, -20, 20))
        y_test_actual = np.expm1(y_test)
        
        test_r2 = r2_score(y_test_actual, preds_test)
        test_mae = mean_absolute_error(y_test_actual, preds_test)
        test_mse = mean_squared_error(y_test_actual, preds_test)
        median_error = np.median(np.abs((preds_test - y_test_actual) / (y_test_actual + 1e-9)) * 100)
        
        results.append({
            'Model': f'Title Only ({name})',
            'R2': test_r2,
            'MAE': test_mae,
            'MSE': test_mse,
            'Median % Error': median_error
        })
    return results[0]

def train_combined_model():
    # Load Data
    data = pd.read_csv(DATASET_PATH, low_memory=False)
    data['title'] = data['title'].astype(str)
    data = data[~data['title'].str.contains('#', regex=False, na=False)]

    # Feature Engineering
    data['publishedAt'] = pd.to_datetime(data['publishedAt'], errors='coerce')
    data['pub_year'] = data['publishedAt'].dt.year
    data['pub_month'] = data['publishedAt'].dt.month
    data['pub_day'] = data['publishedAt'].dt.day
    data['pub_weekday'] = data['publishedAt'].dt.weekday
    data['pub_hour'] = data['publishedAt'].dt.hour
    data['days_since_first_video'] = (data['publishedAt'] - data['publishedAt'].min()).dt.days

    def duration_to_seconds(d):
        if not isinstance(d, str): return 0
        h = int(re.search(r'(\d+)H', d).group(1)) if 'H' in d else 0
        m = int(re.search(r'(\d+)M', d).group(1)) if 'M' in d else 0
        s = int(re.search(r'(\d+)S', d).group(1)) if 'S' in d else 0
        return h * 3600 + m * 60 + s

    data['duration_sec'] = data['duration'].apply(duration_to_seconds)

    sid = SentimentIntensityAnalyzer()
    data['sentiment'] = data['title'].apply(lambda x: sid.polarity_scores(x)['compound'])
    data['readability'] = data['title'].apply(textstat.flesch_reading_ease)
    data['title_length'] = data['title'].apply(len)
    data['word_count'] = data['title'].apply(lambda x: len(x.split()))
    data['uppercase_word_count'] = data['title'].apply(lambda x: len([w for w in x.split() if w.isupper()]))
    data['emoji_count'] = data['title'].apply(emoji.emoji_count)
    data['punctuation_count'] = data['title'].apply(lambda x: len([c for c in x if c in '.,?!:;']))
    data['has_question_mark'] = data['title'].apply(lambda x: '?' in x).astype(int)
    data['has_exclamation_mark'] = data['title'].apply(lambda x: '!' in x).astype(int)

    key_features = ['viewCount', 'likeCount', 'commentCount', 'pub_year']
    data.dropna(subset=key_features, inplace=True)
    data.drop_duplicates(subset='video_id', inplace=True, ignore_index=True)
    data = data[data['viewCount'] >= 1000]
    data['log_views'] = np.log1p(data['viewCount'])

    # Train-Test Split
    train_df, temp_df = train_test_split(data, test_size=0.3, random_state=42)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

    # Preprocessing
    if 'channel_id' in train_df.columns:
        ch_stats = train_df.groupby('channel_id')['viewCount'].agg(['mean', 'median', 'std', 'count']).fillna(0)
        ch_stats.columns = [f'channel_{c}_views' for c in ch_stats.columns]
        train_df = train_df.merge(ch_stats, on='channel_id', how='left')
        val_df = val_df.merge(ch_stats, on='channel_id', how='left')
        test_df = test_df.merge(ch_stats, on='channel_id', how='left')
        for col in ch_stats.columns:
            val_df[col] = val_df[col].fillna(0)
            test_df[col] = test_df[col].fillna(0)
    else:
        for c in ['mean', 'median', 'std', 'count']:
            col_name = f'channel_{c}_views'
            train_df[col_name] = 0
            val_df[col_name] = 0
            test_df[col_name] = 0

    tfidf_vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
    tfidf_train = tfidf_vectorizer.fit_transform(train_df['title'])
    tfidf_val = tfidf_vectorizer.transform(val_df['title'])
    tfidf_test = tfidf_vectorizer.transform(test_df['title'])

    numeric_features = [
        'duration_sec','pub_year','pub_month','pub_day','pub_weekday','pub_hour',
        'days_since_first_video','likeCount','commentCount',
        'channel_mean_views','channel_median_views','channel_std_views',
        'channel_count_views','sentiment','readability','title_length','word_count',
        'uppercase_word_count','emoji_count','punctuation_count',
        'has_question_mark','has_exclamation_mark'
    ]
    cat_cols = [c for c in ['definition','caption','category'] if c in data.columns]

    train_df = pd.get_dummies(train_df, columns=cat_cols, drop_first=True)
    val_df = pd.get_dummies(val_df, columns=cat_cols, drop_first=True)
    test_df = pd.get_dummies(test_df, columns=cat_cols, drop_first=True)

    train_cols = train_df.columns
    val_df = val_df.reindex(columns=train_cols, fill_value=0)
    test_df = test_df.reindex(columns=train_cols, fill_value=0)

    encoded_cat_cols = [c for c in train_df.columns if any(col in c for col in cat_cols)]
    feature_cols = numeric_features + encoded_cat_cols

    X_train_num = train_df[feature_cols].fillna(0)
    X_val_num = val_df[feature_cols].fillna(0)
    X_test_num = test_df[feature_cols].fillna(0)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_num)
    X_val_scaled = scaler.transform(X_val_num)
    X_test_scaled = scaler.transform(X_test_num)

    X_train = hstack([tfidf_train, X_train_scaled])
    X_val = hstack([tfidf_val, X_val_scaled])
    X_test = hstack([tfidf_test, X_test_scaled])

    y_train = train_df['log_views']
    y_val = val_df['log_views']
    y_test = test_df['log_views']

    # Train and Evaluate Models
    models = {
        "GradientBoosting": GradientBoostingRegressor(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=42)
    }

    results = []
    for name, model in models.items():
        model.fit(X_train, y_train)
        preds_test_log = model.predict(X_test)
        preds_test = np.expm1(np.clip(preds_test_log, -20, 20))
        y_test_actual = np.expm1(y_test)
        
        test_r2 = r2_score(y_test_actual, preds_test)
        test_mae = mean_absolute_error(y_test_actual, preds_test)
        test_mse = mean_squared_error(y_test_actual, preds_test)
        median_error = np.median(np.abs((preds_test - y_test_actual) / (y_test_actual + 1e-9)) * 100)
        
        results.append({
            'Model': f'Combined ({name})',
            'R2': test_r2,
            'MAE': test_mae,
            'MSE': test_mse,
            'Median % Error': median_error
        })
    return results[0]

# %% === Run Models and Compare ===
tabular_results = train_tabular_model()
title_results = train_title_model()
combined_results = train_combined_model()

comparison_df = pd.DataFrame([tabular_results, title_results, combined_results])
print(comparison_df)

# %% === Visualization ===
plt.figure(figsize=(12, 6))
sns.barplot(x='R2', y='Model', data=comparison_df, palette='viridis')
plt.title('Model R2 Score Comparison')
plt.xlabel('R2 Score')
plt.ylabel('Model')
plt.show()

plt.figure(figsize=(12, 6))
sns.barplot(x='MAE', y='Model', data=comparison_df.sort_values('MAE', ascending=True), palette='plasma')
plt.title('Model MAE Comparison')
plt.xlabel('Mean Absolute Error')
plt.ylabel('Model')
plt.show()

plt.figure(figsize=(12, 6))
sns.barplot(x='MSE', y='Model', data=comparison_df.sort_values('MSE', ascending=True), palette='magma')
plt.title('Model MSE Comparison')
plt.xlabel('Mean Squared Error')
plt.ylabel('Model')
plt.show()

plt.figure(figsize=(12, 6))
sns.barplot(x='Median % Error', y='Model', data=comparison_df.sort_values('Median % Error', ascending=True), palette='cividis')
plt.title('Model Median % Error Comparison')
plt.xlabel('Median % Error')
plt.ylabel('Model')
plt.show()

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/mohamedhagali/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
